# CSC_51054 - Influencer or Observer: Predicting Social Roles

Aziz Berthé, Octave Rebourseau and Arthur Fournier

### Importing Libraries

In [1]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Libs for TF-IDF
from collections import Counter
import re
import emoji
import nltk

# Libs for transformer models
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# Libraries for XGBoost
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, LabelEncoder, MaxAbsScaler

# Libraries for encoder (CamemBERT)
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin


We download the models and data required by some libraries

# 1. Data Loading

In [2]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

In [3]:
# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

# 2. Data pre-processing
Cleaning the raw entry

In [4]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
X_train['full_text'] = X_train.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
X_kaggle['full_text'] = X_kaggle.apply(lambda tweet: extract_full_text(tweet), axis=1)

### Cleaning the text

In [5]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'https?://(www\.)?', '', text)  # normalize URLs
    text = emoji.demojize(text, language='fr')
    text = text.replace(":", " ").replace("_", " ")
    return text


X_train['clean_bio'] = X_train['user.description'].apply(clean_text)
X_train['clean_text'] = X_train['full_text'].apply(clean_text)
X_kaggle['clean_bio'] = X_kaggle['user.description'].apply(clean_text)
X_kaggle['clean_text'] = X_kaggle['full_text'].apply(clean_text)

### User features

In [6]:
def preprocess_user_features(df):
    df = df.copy()

    df['user.created_at'] = pd.to_datetime(df['user.created_at'], format='%a %b %d %H:%M:%S +0000 %Y')
    ref_date = df['user.created_at'].max()
    df['user.account_age_days'] = (ref_date - df['user.created_at']).dt.days
    df['user.account_age_days_log'] = np.log1p(df['user.account_age_days'])

    def clean_url(url):
        if pd.isna(url) or url == 'None':
            return ""
        clean = re.sub(r'https?://(www\.)?', '', url)
        clean = clean.split('/')[0]
        return clean.replace('.', ' ')

    df['user.url_text'] = df['user.url'].apply(clean_url)

    df['user.has_profile_image'] = df['user.profile_image_url'].notna() & (df['user.profile_image_url'] != '')
    df['user.has_background_image'] = df['user.profile_background_image_url'].notna() & (df['user.profile_background_image_url'] != '')
    
    df['user.location'] = df['user.location'].fillna('').astype(str)
    return df


In [7]:
X_train = preprocess_user_features(X_train)
X_kaggle = preprocess_user_features(X_kaggle)

### Time Features

In [8]:
def extract_date_features(df, col_name):
    """
    Converts date features into more useful formats
    """
    temp_date = pd.to_datetime(df[col_name], errors='coerce')
    df[f'{col_name}.day_of_week'] = temp_date.dt.day_name()    
    df[f'{col_name}.time_of_day'] = pd.cut(
        temp_date.dt.hour, 
        bins=[-1, 5, 11, 17, 23, 24], 
        labels=['Night', 'Morning', 'Afternoon', 'Evening', 'Night'],
        ordered=False
    )
    df[f'{col_name}.exact_time'] = (
        temp_date.dt.hour
    )
    return df

def calculate_time_delta(df, col_a, col_b, new_col_name='delta_temps'):
    """
    Calculates the time difference between two previously processed date columns.
    """
    if f'{col_a}.exact_time' in df.columns and f'{col_b}.exact_time' in df.columns:
        df[new_col_name] = df[f'{col_a}.exact_time'] - df[f'{col_b}.exact_time']
    else:
        df[new_col_name] = np.nan
    return df

def full_date_preprocessing(df):
    """
    Master function to run the full date pipeline.
    """
    df = (
        df
        .pipe(extract_date_features, col_name='created_at')
        .pipe(extract_date_features, col_name='quoted_status.created_at')
        .pipe(calculate_time_delta, col_a='quoted_status.created_at', col_b='created_at')
    )
    return df

X_train = full_date_preprocessing(X_train)
X_kaggle = full_date_preprocessing(X_kaggle)

C:\Users\Octave\AppData\Local\Temp\ipykernel_2724\75197936.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col_name], errors='coerce')
C:\Users\Octave\AppData\Local\Temp\ipykernel_2724\75197936.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col_name], errors='coerce')


### Other features

In [9]:
from datetime import datetime

def calculate_account_age(row):
    try:
        tweet_date = pd.to_datetime(row['created_at'])
        user_date = pd.to_datetime(row['user.created_at'])
        return (tweet_date - user_date).days
    except:
        return np.nan

# Add the tweet length feature
X_train['text_length'] = X_train['full_text'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)
X_kaggle['text_length'] = X_kaggle['full_text'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)

# Add the account age feature
X_train['account_age_days'] = X_train.apply(calculate_account_age, axis=1)
X_kaggle['account_age_days'] = X_kaggle.apply(calculate_account_age, axis=1)

# Add number of hashtags and mentions features
X_train['num_hashtags'] = X_train['entities.hashtags'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)
X_kaggle['num_hashtags'] = X_kaggle['entities.hashtags'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# Add number of mentions feature
X_train['num_mentions'] = X_train['entities.user_mentions'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)
X_kaggle['num_mentions'] = X_kaggle['entities.user_mentions'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# 3. Feature Analysis & Selection



### a. Explore our data
Let's dive into our data, by extracting just a few sample to see at the used variables

In [10]:
nb_extracted = 20   # We just need a few samples for testing

subset = []
with open("train.jsonl", "r") as f:
    for i, line in enumerate(f):
        if i >= nb_extracted:
            break
        subset.append(json.loads(line))

# Let's save it into a .json file for visualisation
with open("data_subset.json", "w") as f:
    f.write(json.dumps(subset, indent=4))

Let's take a look more generally at the different labels, especially their names and data type.

In [11]:
types = set()
for col in X_train.columns:
    print(f"{col}: \tType={X_train[col].dtype}, \tEmptiness rate={X_train[col].isnull().mean():.2f}")
    types.add(X_train[col].dtype)
print(f"Types de données dans le dataset : {types}")

in_reply_to_status_id_str: 	Type=float64, 	Emptiness rate=0.70
in_reply_to_status_id: 	Type=float64, 	Emptiness rate=0.70
created_at: 	Type=datetime64[ns, UTC], 	Emptiness rate=0.00
in_reply_to_user_id_str: 	Type=float64, 	Emptiness rate=0.69
source: 	Type=object, 	Emptiness rate=0.00
quoted_status_id: 	Type=float64, 	Emptiness rate=0.65
retweet_count: 	Type=int64, 	Emptiness rate=0.00
retweeted: 	Type=bool, 	Emptiness rate=0.00
geo: 	Type=float64, 	Emptiness rate=1.00
filter_level: 	Type=object, 	Emptiness rate=0.00
in_reply_to_screen_name: 	Type=object, 	Emptiness rate=0.69
is_quote_status: 	Type=bool, 	Emptiness rate=0.00
id_str: 	Type=int64, 	Emptiness rate=0.00
in_reply_to_user_id: 	Type=float64, 	Emptiness rate=0.69
favorite_count: 	Type=int64, 	Emptiness rate=0.00
text: 	Type=object, 	Emptiness rate=0.00
place: 	Type=float64, 	Emptiness rate=1.00
lang: 	Type=object, 	Emptiness rate=0.00
quote_count: 	Type=int64, 	Emptiness rate=0.00
favorited: 	Type=bool, 	Emptiness rate=0.00
co

### b. Feature selection
We're gonna reduce the number of features by only selecting the most relevant ones, to prevent the model from using useless parameters.

Empty columns

In [12]:
print(f"Number of features before cleanning: {len(X_train.columns)}")

# We only want to drop the columns that are have more than the threshold of NaN values
nan_threshold = 1.0  # This means we remove all columns that are completely empty

features = [c for c in X_train.columns if X_train[c].isnull().mean() < nan_threshold]
not_selected_features = [c for c in X_train.columns if c not in features]
print(f"Dropping {len(not_selected_features)} empty features. {len(features)} features remain.")

Number of features before cleanning: 211
Dropping 25 empty features. 186 features remain.


Deleting constant features


In [13]:

Val_Unique = X_train.copy()
for col in X_train.select_dtypes(include=['object']).columns:
    Val_Unique[col] = X_train[col].astype(str)

unique_counts = Val_Unique.nunique()
cols_to_drop = unique_counts[unique_counts <= 1].index.tolist()

print(f"Deleting {len(cols_to_drop)} columns...")
X_train = X_train.drop(columns=cols_to_drop)
features = [c for c in features if c not in cols_to_drop]
# Affiche quelques exemples pour vérifier
print("Examples :", cols_to_drop[:5])

Deleting 37 columns...
Examples : ['retweet_count', 'retweeted', 'geo', 'filter_level', 'favorite_count']


In [14]:
# We now drop these columns we consider not useful for modeling
feature_to_exclude = {'challenge_id','withheld_in_countries','label','text','user.description'}

# Re-select features
features = [f for f in features if f not in feature_to_exclude]
print(f"Dropping {len(feature_to_exclude)} features. {len(features)} features remain.")

Dropping 5 features. 170 features remain.


In [15]:
numeric_features = X_train[features].select_dtypes(include=['number']).columns.tolist()


categorical_features = [
    'is_quote_status',
    'user.default_profile_image', 
    'user.profile_use_background_image',
    'user.default_profile' 
]

user_features = [i for i in features if 'user.' in i and 'quoted' not in i]+[
    'user.account_age_days',
    'user.account_age_days_log',
    'user.url_text',
    'user.has_background_image',
    'clean_bio'
]
drop_features = [
    'full_text',
    'text',
    'user.description',
    'created_at',
    # ...
]
categorical_features = [c for c in features if (c not in numeric_features) and (c not in user_features) and (c not in drop_features) and (c != 'clean_text')]

### c. Creating training and validation sets
Avoiding data leakage between different tweets from same user

In [16]:
df = X_train.copy()
df['target'] = y_train
groups = df['user.created_at'] 
X = df.drop('target', axis=1)
y = df['target']

splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)

train_idx, val_idx = next(splitter.split(X, y, groups))

X_tr = X.iloc[train_idx]
y_tr = y.iloc[train_idx]

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

print(f"Nombre de tweets dans Train : {len(X_tr)}")
print(f"Nombre de tweets dans Val   : {len(X_val)}")

Nombre de tweets dans Train : 123903
Nombre de tweets dans Val   : 31011


# 4. Implementing the 3 different predictions

## Pipeline 1 - TF-IDF

#### useful functions

In [17]:
nltk.download('stopwords')
french_stop_words = stopwords.words('french')

# Cleaning functions to avoid issues
def cast_text_to_string(x):
    return x.astype(str).fillna('')

def cast_categorical_to_string(x):
    # Convert ['a', 'b'] lists into "['a', 'b']" strings so they can be accepted
    return x.astype(str)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Octave\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### The core pipeline

In [18]:
text_transformer_TFIDF = Pipeline([
    ('caster', FunctionTransformer(cast_text_to_string, validate=False)),
    ('tfidf', TfidfVectorizer(
        stop_words=french_stop_words,
        max_df=0.7,      
        min_df=2,        
        max_features=2000,
        ngram_range=(1, 3) 
    ))
])

categorical_transformer = Pipeline([
    ('caster', FunctionTransformer(cast_categorical_to_string, validate=False)),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=True, 
        min_frequency=0.005,
        max_categories=150  
    ))
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MaxAbsScaler())
])

preprocessor = ColumnTransformer([
    ('text', text_transformer_TFIDF, 'clean_text'),
    ('cat', categorical_transformer, categorical_features),
    ('num', numeric_transformer, numeric_features)
], remainder='drop')

model_pipeline_TFIDF = Pipeline([
    ('pre', preprocessor),
    ('clf', XGBClassifier(
        n_estimators= 650, 
        learning_rate= 0.04, 
        max_depth= 6, 
        min_child_weight= 4, 
        subsample= 0.7,
        colsample_bytree= 0.95,
        gamma=0.2,
        reg_alpha= 0.05, 
        reg_lambda= 2.3,

        tree_method='hist',     
        device='cpu',           
        n_jobs=-1,            
        random_state=42,
        eval_metric='logloss',

        
    ))
])

### Training

In [19]:
model_pipeline_TFIDF.fit(X_tr, y_tr)
tfidf_pred = model_pipeline_TFIDF.predict(X_val)
print(f"Accuracy TF-IDF pipeline: {accuracy_score(y_val, tfidf_pred):.4f}")

Accuracy TF-IDF pipeline: 0.8354


### Predictions

In [20]:
# Prediction
y_pred_test = model_pipeline_TFIDF.predict(X_kaggle[features])

output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_test)], axis=1, ignore_index=True)
output.columns = ['ID', 'Prediction'] # Expected names by Kaggle
output.to_csv('XGBoost_TF-IDF.csv', index=False)

## Pipeline 2 - Using BERT

### Camembert

Let's generate embedding once

In [21]:
def generate_bert_embeddings(df, text_col, model_name='vinai/bertweet-base', batch_size=32, max_length=128):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"--- Loading {model_name} on {device} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    model.to(device)
    model.eval()

    texts = df[text_col].fillna("").astype(str).tolist()
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i : i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)

    final_embeddings = np.vstack(all_embeddings)
    cols = [f'bert_{i}' for i in range(final_embeddings.shape[1])]

    df_emb = pd.DataFrame(final_embeddings, columns=cols, index=df.index)
    return df_emb, cols

In [22]:
# For user description first :
bertBio_tr, bert_cols_bio = generate_bert_embeddings(X_train, 'clean_bio',max_length=64)
bertBio_tr = bertBio_tr.add_prefix('bio_') 
X_train = pd.concat([X_train, bertBio_tr], axis=1)

bertBio_kaggle, _ = generate_bert_embeddings(X_kaggle, 'clean_bio',max_length=64)
bertBio_kaggle = bertBio_kaggle.add_prefix('bio_') 
X_kaggle = pd.concat([X_kaggle, bertBio_kaggle], axis=1)

bert_cols_bio = [f'bio_{col}' for col in bert_cols_bio]

# For the tweet text :
df_bertText_tr, bert_cols_text = generate_bert_embeddings(X_train, 'clean_text',max_length=96)
df_bertText_tr = df_bertText_tr.add_prefix('text_') 
X_train = pd.concat([X_train, df_bertText_tr], axis=1)

df_bertText_kaggle, _ = generate_bert_embeddings(X_kaggle, 'clean_text',max_length=96) 
df_bertText_kaggle = df_bertText_kaggle.add_prefix('text_')
X_kaggle = pd.concat([X_kaggle, df_bertText_kaggle], axis=1)

bert_cols_text = [f'text_{col}' for col in bert_cols_text]

--- Loading vinai/bertweet-base on cuda ---


100%|██████████| 4842/4842 [02:09<00:00, 37.51it/s]


--- Loading vinai/bertweet-base on cuda ---


100%|██████████| 3231/3231 [01:27<00:00, 36.91it/s]


--- Loading vinai/bertweet-base on cuda ---


100%|██████████| 4842/4842 [03:35<00:00, 22.46it/s]


--- Loading vinai/bertweet-base on cuda ---


100%|██████████| 3231/3231 [02:25<00:00, 22.19it/s]


In [23]:
df = X_train.copy()
df['target'] = y_train
groups = df['user.created_at'] 
X = df.drop('target', axis=1)
y = df['target']

splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)

train_idx, val_idx = next(splitter.split(X, y, groups))

X_tr = X.iloc[train_idx]
y_tr = y.iloc[train_idx]

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

print(f"Number of tweets in Train : {len(X_tr)}")
print(f"Number of tweets in Val   : {len(X_val)}")

Number of tweets in Train : 123903
Number of tweets in Val   : 31011


### Core elements

In [24]:
categorical_transformer = Pipeline([
    ('caster', FunctionTransformer(cast_categorical_to_string, validate=False)),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=True, 
        min_frequency=0.005, 
        max_categories=150
    ))
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MaxAbsScaler())
])

preprocessor = ColumnTransformer([
    ('text', 'passthrough', bert_cols_text),
    ('bio', 'passthrough', bert_cols_bio),
    ('cat', categorical_transformer, categorical_features),
    ('num', numeric_transformer, numeric_features)
], remainder='drop')

model_pipeline_BERT = Pipeline([
    ('pre', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=4,
        subsample=0.9,
        colsample_bytree=0.75,
        reg_alpha=0.1,
        reg_lambda=0.3,
        gamma=0.2,
        
        tree_method='hist',
        device='cpu',
        n_jobs=-1,
        random_state=42,
         eval_metric='logloss'
    ))
])

### Training

In [25]:
model_pipeline_BERT.fit(X_tr, y_tr)
BERT_pred = model_pipeline_BERT.predict(X_val)
print(f"Accuracy BERT pipeline: {accuracy_score(y_val, BERT_pred):.4f}")

Accuracy BERT pipeline: 0.8341


### Predictions

In [27]:
# Prediction
y_pred_test = model_pipeline_BERT.predict(X_kaggle)

output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_test)], axis=1, ignore_index=True)
output.columns = ['ID', 'Prediction'] # Expected names by Kaggle
output.to_csv('XGBoost_BERT.csv', index=False)

## Pipeline 3 - Specialized model on user profiles

### Core element

In [28]:
tfidf_location = Pipeline([
    ('caster', FunctionTransformer(cast_text_to_string, validate=False)),
    ('tfidf', TfidfVectorizer(    
        max_features=15,
        ngram_range=(1, 2) 
    ))
])
tfidf_url = Pipeline([
    ('caster', FunctionTransformer(cast_text_to_string, validate=False)),
    ('tfidf', TfidfVectorizer(
        max_features=20,
        ngram_range=(1, 1) 
    ))
])
textbio_transformer_TFIDF = Pipeline([
    ('caster', FunctionTransformer(cast_text_to_string, validate=False)),
    ('tfidf', TfidfVectorizer(
        stop_words=french_stop_words,
        max_df=0.7,      
        min_df=2,         
        max_features=750, 
        ngram_range=(1, 2) 
    ))
])

preprocessor_users = ColumnTransformer(
    transformers=[
        ('desc_tfidf',textbio_transformer_TFIDF, 'clean_bio'),
        ('bert_pass', 'passthrough', bert_cols_bio),

        ('loc_tfidf', tfidf_location, 'user.location'),        
        ('url_tfidf', tfidf_url, 'user.url_text'),
        
        ('num_scaler', StandardScaler(), [
            'user.account_age_days_log',
            'user.account_age_days',
            'user.favourites_count',
            'user.listed_count'
        ]),
        
        ('bool_pass', 'passthrough', [
            'user.has_background_image',
            'user.default_profile',
            'user.is_translator',
        ])
    ],
    remainder='drop' 
)
user_profile_model = Pipeline(steps=[
    ('preprocessor', preprocessor_users),
    ('clf', XGBClassifier(
        n_estimators=900,
        learning_rate=0.015,
        max_depth=7,
        device='cpu',
        n_jobs=-1,
        random_state=42,
        tree_method='hist',
        eval_metric='logloss', 
        objective='binary:logistic',
        min_child_weight=2,
        reg_alpha=0.01,
        reg_lambda=0.1, 
        subsample=0.8,
        colsample_bytree=0.6
    ))
])

### Preparing train/val set with unique user

In [29]:
train_temp = X_tr.copy()
train_temp['target_label'] = y_tr

df_unique_train = train_temp.groupby('user.created_at').last().reset_index()

X_tr_user= df_unique_train.drop(columns=['target_label'])
y_tr_user = df_unique_train['target_label']

val_temp = X_val.copy()
val_temp['target_label'] = y_val
df_unique_val = val_temp.groupby('user.created_at').last().reset_index()
X_val_user= df_unique_val.drop(columns=['target_label'])
y_val_user = df_unique_val['target_label']

### Training

In [30]:
user_profile_model.fit(X_tr_user, y_tr_user)
Profile_pred = user_profile_model.predict(X_val)
print(f"Accuracy BERT pipeline: {accuracy_score(y_val, Profile_pred):.4f}")

Accuracy BERT pipeline: 0.8155


### Predictions

In [32]:
# Prediction
y_pred_test = user_profile_model.predict(X_kaggle)

output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_test)], axis=1, ignore_index=True)
output.columns = ['ID', 'Prediction'] # Expected names by Kaggle
output.to_csv('XGBoost_TF-IDF.csv', index=False)

# 5. Combining models

### Calculating probabilities

In [33]:
BERT_prob = model_pipeline_BERT.predict_proba(X_val)[:,1]
TFIDF_prob = model_pipeline_TFIDF.predict_proba(X_val)[:,1]
Profile_prob = user_profile_model.predict_proba(X_val)[:,1]

### Combining prediction 
Weighted vote system outputting a single label for a given user (some users having several tweets)

In [34]:
weights =[0.3,0.5,0.2]

ids_users = X_val['user.created_at']

df_scores = pd.DataFrame({
    'user_id': ids_users,
    'BERT': BERT_prob,
    'TFIDF': TFIDF_prob,
    'Profile' :Profile_prob

})

BERT_agg = df_scores.groupby('user_id')['BERT'].mean()
TFIDF_agg = df_scores.groupby('user_id')['TFIDF'].mean()
Profile_agg = df_scores.groupby('user_id')['Profile'].mean()

final_df = pd.concat([BERT_agg,TFIDF_agg,Profile_agg], axis=1)

final_df['final_proba'] = (
    (final_df['BERT'] * weights[0]) + 
    (final_df['TFIDF'] * weights[1])+
    (final_df['Profile']* weights[2])
    )

final_df['prediction'] = (final_df['final_proba'] > 0.5).astype(int)

y_pred_aggregated = ids_users.map(final_df['prediction'] )


acc = accuracy_score(y_val, y_pred_aggregated)



print(f"Accuracy : {acc:.4f}")

Accuracy : 0.8409


# 6. Submission

## Train on full dataset

In [35]:
model_pipeline_BERT.fit(X_train,y_train)
model_pipeline_TFIDF.fit(X_train,y_train)

temp = X_train.copy()
temp['target_label'] = y_tr
unique_train = train_temp.groupby('user.created_at').last().reset_index()
X_train_user= unique_train.drop(columns=['target_label'])
y_train_user = unique_train['target_label']


user_profile_model.fit(X_train_user,y_train_user)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('desc_tfidf',
                                                  Pipeline(steps=[('caster',
                                                                   FunctionTransformer(func=<function cast_text_to_string at 0x00000288E055B4C0>)),
                                                                  ('tfidf',
                                                                   TfidfVectorizer(max_df=0.7,
                                                                                   max_features=750,
                                                                                   min_df=2,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words=['au',
                                                                                               'aux',
                                                                                               'avec',
                                                                                               'ce',
                                                                                               'ces',
                                                                                               'dans',
                                                                                               'de',
                                                                                               'des',
                                                                                               'du',
                                                                                               'elle',
                                                                                               'en',
                                                                                               'et',
                                                                                               'eux',
                                                                                               'i...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.015, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=7,
                               max_leaves=None, min_child_weight=2, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=900, n_jobs=-1,
                               num_parallel_tree=None, ...))])

### Calculating probabilities

In [36]:
BERT_prob = model_pipeline_BERT.predict_proba(X_kaggle)[:,1]
TFIDF_prob = model_pipeline_TFIDF.predict_proba(X_kaggle)[:,1]
Profile_prob = user_profile_model.predict_proba(X_kaggle)[:,1]

### Combining results

In [37]:
ids_users = X_kaggle['user.created_at']

df_scores = pd.DataFrame({
    'user_id': ids_users,
    'BERT': BERT_prob,
    'TFIDF': TFIDF_prob,
    'Profile' :Profile_prob

})

BERT_agg = df_scores.groupby('user_id')['BERT'].mean()
TFIDF_agg = df_scores.groupby('user_id')['TFIDF'].mean()
Profile_agg = df_scores.groupby('user_id')['Profile'].mean()

final_df = pd.concat([BERT_agg,TFIDF_agg,Profile_agg], axis=1)

final_df['final_proba'] = (
    (final_df['BERT'] * weights[0]) + 
    (final_df['TFIDF'] * weights[1])+
    (final_df['Profile']* weights[2])
    )

final_df['prediction'] = (final_df['final_proba'] > 0.5).astype(int)

y_pred_aggregated = ids_users.map(final_df['prediction'] )

### Creating sub file

In [38]:
output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_aggregated)], axis=1, ignore_index=True)
output.columns = ['ID', 'Prediction']
output.to_csv('final_submission.csv', index=False)